## 🔷 Step 1: Define a Neural Network–Style Function

Let’s use a **simple 2-layer MLP**:

$$
\begin{align*}
x &= \begin{bmatrix} x_1 \\ x_2 \end{bmatrix} \in \mathbb{R}^2 \\
W_1 &= \begin{bmatrix} 1 & -1 \\ 2 & 0 \end{bmatrix},\quad b_1 = \begin{bmatrix} 0 \\ 1 \end{bmatrix} \\
W_2 &= \begin{bmatrix} 1 & 2 \end{bmatrix},\quad b_2 = 0
\end{align*}
$$

Function:

$$
z = f(x) = W_2 \cdot \text{ReLU}(W_1x + b_1) + b_2
$$

We’ll manually go through:

* `Linear → ReLU → Linear`

---

## 🔷 Step 2: Forward Pass

Let input be:

$$
x = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix}
$$

### First Layer (Affine):

$$
a_1 = W_1 x + b_1 = 
\begin{bmatrix} 1 & -1 \\ 2 & 0 \end{bmatrix}
\begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix}
+
\begin{bmatrix} 0 \\ 1 \end{bmatrix}
=
\begin{bmatrix} 1 - 2 \\ 2 \cdot 1 \end{bmatrix} +
\begin{bmatrix} 0 \\ 1 \end{bmatrix}
=
\begin{bmatrix} -1 \\ 3 \end{bmatrix}
$$

### ReLU Activation:

$$
h = \text{ReLU}(a_1) = \max(0, a_1) = \begin{bmatrix} 0 \\ 3 \end{bmatrix}
$$

### Second Layer:

$$
z = W_2 h + b_2 = \begin{bmatrix} 1 & 2 \end{bmatrix} \cdot \begin{bmatrix} 0 \\ 3 \end{bmatrix} + 0 = 0 + 6 = 6
$$

---

## 🔷 Step 3: Computational Graph

```
         x1       x2
        │        │
        └──┬─────┘
           ▼
        [ W1x + b1 ]         ← Linear Layer 1
           │
        ┌──┴──────────────┐
        ▼                 ▼
      a1_1              a1_2
        │                 │
    ReLU()             ReLU()
        │                 │
        ▼                 ▼
      h1                h2
        └──────┬─────────┘
               ▼
         [ W2 · h + b2 ]         ← Linear Layer 2
               │
               ▼
               z

```

Intermediate values:

* $a_1 = [-1, 3]$
* $h = [0, 3]$
* $z = 6$

---

## 🔷 Step 4: Backward Pass (Manual Chain Rule)

We want:

$$
\frac{∂z}{∂x_1},\quad \frac{∂z}{∂x_2}
$$

We'll go backward, storing gradients at each node.

---

### 🔹 Step-by-Step Derivatives:

Let’s denote:

* $a = W_1 x + b_1$
* $h = \text{ReLU}(a)$
* $z = W_2 h$

---

### 🔸 ∂z/∂h:

Since:

$$
z = w_2^T h = w_{21} h_1 + w_{22} h_2
\Rightarrow \frac{∂z}{∂h_1} = w_{21} = 1,\quad \frac{∂z}{∂h_2} = w_{22} = 2
$$

---

### 🔸 ∂h/∂a (ReLU Gradients):

We had:

$$
a = [-1, 3] \Rightarrow h = [0, 3]
$$

* ReLU′(-1) = 0
* ReLU′(3) = 1

So:

$$
\frac{∂h_1}{∂a_1} = 0,\quad \frac{∂h_2}{∂a_2} = 1
$$

---

### 🔸 ∂z/∂a:

Using chain rule:

$$
\frac{∂z}{∂a_1} = \frac{∂z}{∂h_1} \cdot \frac{∂h_1}{∂a_1} = 1 \cdot 0 = 0 \\
\frac{∂z}{∂a_2} = \frac{∂z}{∂h_2} \cdot \frac{∂h_2}{∂a_2} = 2 \cdot 1 = 2
$$

---

### 🔸 ∂a/∂x:

Recall:

$$
a = W_1 x + b_1 \Rightarrow
\begin{bmatrix}
a_1 \\
a_2
\end{bmatrix}
=
\begin{bmatrix}
1 & -1 \\
2 & 0
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2
\end{bmatrix}
$$

So,

$$
\frac{∂a_1}{∂x_1} = 1,\quad \frac{∂a_1}{∂x_2} = -1 \\
\frac{∂a_2}{∂x_1} = 2,\quad \frac{∂a_2}{∂x_2} = 0
$$

---

### 🔸 Final Gradients (∂z/∂x):

Using chain rule:

$$
\frac{∂z}{∂x_1} = \frac{∂z}{∂a_1} \cdot \frac{∂a_1}{∂x_1} + \frac{∂z}{∂a_2} \cdot \frac{∂a_2}{∂x_1}
= 0 \cdot 1 + 2 \cdot 2 = 4
$$

$$
\frac{∂z}{∂x_2} = \frac{∂z}{∂a_1} \cdot \frac{∂a_1}{∂x_2} + \frac{∂z}{∂a_2} \cdot \frac{∂a_2}{∂x_2}
= 0 \cdot (-1) + 2 \cdot 0 = 0
$$

---

## ✅ Final Output

```text
Forward:
  x = [1.0, 2.0]
  z = 6.0

Backward:
  dz/dx1 = 4.0
  dz/dx2 = 0.0
```

In [1]:
import torch

x = torch.tensor([1.0, 2.0], requires_grad=True)
W1 = torch.tensor([[1.0, -1.0], [2.0, 0.0]])
b1 = torch.tensor([0.0, 1.0])
W2 = torch.tensor([1.0, 2.0])
b2 = torch.tensor(0.0)

a = W1 @ x + b1      # Linear Layer 1
h = torch.relu(a)    # ReLU Activation
z = W2 @ h + b2      # Linear Layer 2
z.backward()

print(f"z: {z.item()}")
print(f"x.grad: {x.grad}")

z: 6.0
x.grad: tensor([4., 0.])


## 🔷 Model Recap

We are using a simple **2-layer feedforward neural network**:

$$
\begin{align*}
a &= W_1 x + b_1 \quad\text{→ shape } (2,) \\
h &= \text{ReLU}(a) \\
z &= W_2 h + b_2 \quad\text{→ scalar output}
\end{align*}
$$

Where:

* $x \in \mathbb{R}^2$
* $W_1 \in \mathbb{R}^{2 \times 2}$
* $b_1 \in \mathbb{R}^2$
* $W_2 \in \mathbb{R}^{1 \times 2}$
* $b_2 \in \mathbb{R}$

---

## 🔁 Left-to-Right Computational Graph — Now with Parameters

We now include the weights $W_1, b_1, W_2, b_2$ explicitly in the graph:

```
          x1         x2
           │          │
           └────┬─────┘
                ▼
        ┌───────────────┐
        │ Linear Layer  │
        │  a = W1·x + b1│◄──── W1, b1
        └────┬────┬─────┘
             │    │
           a1_1  a1_2
             │    │
          ReLU  ReLU
             │    │
           h1    h2
             └────┬────► W2, b2
                  ▼
          z = W2·h + b2
```

Each parameter is a **leaf tensor** tracked by PyTorch autograd:

* `W1.requires_grad = True`
* `b1.requires_grad = True`
* `W2.requires_grad = True`
* `b2.requires_grad = True`

---

## 🔧 What Happens During `z.backward()`?

After calling `z.backward()`:

* PyTorch applies the **chain rule** and computes gradients:

### 🔹 Gradients w\.r.t. output layer parameters:

Let:

* $z = W_2 h + b_2$
* Then:

$$
\frac{∂z}{∂W_2} = h^T \quad (shape: 2×1) \\
\frac{∂z}{∂b_2} = 1
$$

---

### 🔹 Gradients w\.r.t. hidden layer parameters:

Let:

* $h = \text{ReLU}(a)$
* $a = W_1 x + b_1$

We compute:

$$
\frac{∂z}{∂W_1} = \frac{∂z}{∂h} \cdot \frac{∂h}{∂a} \cdot \frac{∂a}{∂W_1}
$$

Let’s break that:

1. $\frac{∂z}{∂h} = W_2^T$ (gradient back from output)
2. $\frac{∂h}{∂a} = \text{diag}(ReLU'(a))$
3. $\frac{∂a}{∂W_1} = x^T$

So,

$$
\frac{∂z}{∂W_1} = (W_2^T \cdot ReLU'(a)) \cdot x^T
$$

$$
\frac{∂z}{∂b_1} = W_2^T \cdot ReLU'(a)
$$

Each of these will be computed automatically if `requires_grad=True` is set.

## ✅ Summary Table

| Parameter | Gradient Expression                                                                    |
| --------- | -------------------------------------------------------------------------------------- |
| $W_2$     | $\frac{∂z}{∂W_2} = h^T$                                                                |
| $b_2$     | $\frac{∂z}{∂b_2} = 1$                                                                  |
| $W_1$     | $\frac{∂z}{∂W_1} = \delta_h \cdot x^T$, where $\delta_h = W_2^T \cdot \text{ReLU}'(a)$ |
| $b_1$     | $\frac{∂z}{∂b_1} = \delta_h$                                                           |

In [2]:
# Input
x = torch.tensor([1.0, 2.0], requires_grad=False)

# Parameters
W1 = torch.tensor([[1.0, -1.0], [2.0, 0.0]], requires_grad=True)
b1 = torch.tensor([0.0, 1.0], requires_grad=True)
W2 = torch.tensor([1.0, 2.0], requires_grad=True)
b2 = torch.tensor(0.0, requires_grad=True)

# Forward pass
a = W1 @ x + b1      # Linear 1
h = torch.relu(a)    # ReLU
z = W2 @ h + b2      # Linear 2

# Backward
z.backward()

# Gradients w.r.t. parameters
print("dz/dW1:\n", W1.grad)
print("dz/db1:\n", b1.grad)
print("dz/dW2:\n", W2.grad)
print("dz/db2:\n", b2.grad)

dz/dW1:
 tensor([[0., 0.],
        [2., 4.]])
dz/db1:
 tensor([0., 2.])
dz/dW2:
 tensor([0., 3.])
dz/db2:
 tensor(1.)


In [4]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x768ca7d17ca0>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x768ca7d17520>


In [6]:
def print_graph(tensor):
    seen = set()
    def _print(node, indent=0):
        if node in seen:
            return
        seen.add(node)
        print(" " * indent + str(type(node)))
        if hasattr(node, 'next_functions'):
            for n, _ in node.next_functions:
                if n is not None:
                    _print(n, indent + 2)
    _print(tensor.grad_fn)

# Input
x = torch.tensor([1.0, 2.0], requires_grad=False)

# Parameters
W1 = torch.tensor([[1.0, -1.0], [2.0, 0.0]], requires_grad=True)
b1 = torch.tensor([0.0, 1.0], requires_grad=True)
W2 = torch.tensor([1.0, 2.0], requires_grad=True)
b2 = torch.tensor(0.0, requires_grad=True)

# Forward pass
a = W1 @ x + b1      # Linear 1
h = torch.relu(a)    # ReLU
z = W2 @ h + b2      # Linear 2

print_graph(z)

<class 'AddBackward0'>
  <class 'DotBackward0'>
    <class 'AccumulateGrad'>
    <class 'ReluBackward0'>
      <class 'AddBackward0'>
        <class 'MvBackward0'>
          <class 'AccumulateGrad'>
        <class 'AccumulateGrad'>
  <class 'AccumulateGrad'>


### Full Flow Summary
```
z = W2 @ h + b2        ← AddBackward0
├── W2 @ h             ← DotBackward0
│   ├── W2             ← AccumulateGrad(W2)
│   └── h = ReLU(a)    ← ReluBackward0
│       └── a = W1x + b1 ← AddBackward0
│           ├── W1 @ x   ← MvBackward0
│           │   └── W1   ← AccumulateGrad(W1)
│           └── b1       ← AccumulateGrad(b1)
└── b2                 ← AccumulateGrad(b2)

```

## Pytorch Doc Examples

In [ ]:
x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

In [7]:
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.1272, 0.3100, 0.0708],
        [0.1272, 0.3100, 0.0708],
        [0.1272, 0.3100, 0.0708],
        [0.1272, 0.3100, 0.0708],
        [0.1272, 0.3100, 0.0708]])
tensor([0.1272, 0.3100, 0.0708])


In [10]:
z = torch.matmul(x, w) + b
print(z.requires_grad)

True


In [11]:
with torch.no_grad():
    z = torch.matmul(x, w) + b
print(z.requires_grad)

False
